# Introduction to Restricted Boltzmann Machines
In this module, we will explore Boltzmann Machines, a type of stochastic recurrent neural network. Boltzmann machines come in two forms: full Boltzmann machines and restricted Boltzmann machines. The latter is a simplified version that is easier to train and is often used in practice. 

By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:
* __Restricted Boltzmann Machines (RBMs)__ are a type of Boltzmann machine with a bipartite structure, consisting of visible and hidden layers, where each visible unit is connected to all hidden units but not to other visible units, allowing for efficient training via contrastive divergence.
* __Sampling from RBMs:__ The state of an RBM can be sampled by iteratively updating the states of visible and hidden units based on their connections and biases, using a stochastic process that allows for the generation of new data samples from the learned distribution. Wow! This is our first truly generative model!
* __Learning by matching statistics for RBMs:__ Training adjusts weights by gradient ascent on the data log-likelihood—estimating “positive” (data-clamped) and “negative” (free-running) correlations via Gibbs sampling—though this converges slowly in large models, inspiring the popular Restricted Boltzmann Machine (RBM), which uses a bipartite structure for fast contrastive-divergence learning.

This is a super interesting topic, and we will explore it in detail. We will start with the basics of Boltzmann machines and discuss how we train them. Let's dive in!

___

<div>
    <center>
      <img
        src="figs/Fig-DrawSomethingLikeThis-but-Better-Redraw.png"
        alt="triangle with all three sides equal"
        height="600"
        width="300"
      />
    </center>
  </div>

## Restricted Boltzmann Machines (RBMs)
A restricted Boltzmann machine (RBM) is a type of Boltzmann machine that has a bipartite structure, meaning that the nodes can be divided into two disjoint sets: visible nodes and hidden nodes. The visible nodes represent the input data, while the hidden nodes capture the underlying structure of the data (latent variables).
> __How is this different from a Boltzmann machine?__ In a Boltzmann machine, we also have visible and hidden nodes, but all nodes are fully connected to each other, meaning that every node can influence every other node. In an RBM, the visible nodes are only connected to the hidden nodes, and the hidden nodes are only connected to the visible nodes. This means that there are no connections between the visible nodes or between the hidden nodes.

> __Why is this useful?__ The bipartite structure of RBMs allows for efficient training using contrastive divergence, as the number of connections between the visible and hidden nodes is much smaller than in a fully connected Boltzmann machine. This makes RBMs more practical for large datasets and complex models.

Let's look at the formal definition of an RBM. A restricted Boltzmann machine is defined by the tuple $\mathcal{B}_{\text{RBM}} = \left(\mathcal{V}_{\text{vis}},\mathcal{V}_{\text{hid}},\mathcal{E}, \mathbf{W},\mathbf{b}_{\text{vis}}, \mathbf{b}_{\text{hid}}, \mathbf{s}\right)$:
* __Visible Nodes__: The set of visible nodes $\mathcal{V}_{\text{vis}}$ represents the input data. Each visible node $v_{i}\in\mathcal{V}_{\text{vis}}$ has a binary state $s_{i}\in\{-1,1\}$ and a bias term $b_{i}\in\mathbf{b}_{\text{vis}}$. There are $V = |\mathcal{V}_{\text{vis}}|$ visible nodes.
* __Hidden Nodes__: The set of hidden nodes $\mathcal{V}_{\text{hid}}$ captures the underlying structure of the data. Each hidden node $h_{j}\in\mathcal{V}_{\text{hid}}$ has a binary state $s_{j}\in\{-1,1\}$ and a bias term $b_{j}\in\mathbf{b}_{\text{hid}}$. There are $H = |\mathcal{V}_{\text{hid}}|$ hidden nodes.
* __Edges__: The set of edges $\mathcal{E}$ connects the visible nodes to the hidden nodes. Each edge $e_{ij}\in\mathcal{E}$ connects a visible node $v_{i}\in\mathcal{V}_{\text{vis}}$ to a hidden node $h_{j}\in\mathcal{V}_{\text{hid}}$. The weight of the edge connecting $v_{i}$ and $h_{j}$ is denoted by $w_{ij}\in\mathbf{W}$, where the weight matrix $\mathbf{W}\in\mathbb{R}^{V\times{H}}$ is symmetric, i.e. $w_{ij} = w_{ji}$ and $w_{ii} = 0$ (no self loops). The weights $w_{ij}\in\mathbb{R}$ determine the strength of the connection between nodes $i$ and $j$. 
* __States__: The state of the RBM $\mathcal{B}_{\text{RBM}}$ is represented by a binary vector $\mathbf{s}\in\mathbb{R}^{|\mathcal{V}_{\text{vis}}| + |\mathcal{V}_{\text{hid}}|}$, where $s_{i}\in\{-1,1\}$ is the state of node $v_{i}$ and $s_{j}\in\{-1,1\}$ is the state of node $h_{j}$. The set of all possible _state configurations_ is denoted by $\mathcal{S} \equiv \left\{\mathbf{s}^{(1)},\mathbf{s}^{(2)},\ldots,\mathbf{s}^{(N)}\right\}$, where $N$ is the number of possible state configurations, or $N = 2^{V+H}$ for binary units.


Now that have formally defined the machine $\mathcal{B}_{\text{RBM}}$, how can we use it, i.e., how can we draw samples from it?

## Restricted Boltzmann Dynamics
Suppose we let the state of a Restricted Boltzmann Machine $\mathcal{B}_{\text{RBM}}$ evolve over $t=1,2,\dots, T$ turns, where the state of each node at turn $t$ is binary $s_{i}^{(t)} \in \{-1, 1\}$. During each turn, every visible and hidden node can update its state based on the states of the nodes its connected to, the weights of its connections, and its bias term. 
> __Connections:__ In a Restricted Boltzmann Machine, visible nodes are only connected to the hidden nodes, and vice-versa, and there are no connections between the visible nodes or between the hidden nodes. Thus, there will be $V\times{H}$ connections in total, where $V$ is the number of visible nodes and $H$ is the number of hidden nodes.

Let the nodes in the RBM be denoted by $\mathcal{V} = \mathcal{V}_{\text{vis}}\cup\mathcal{V}_{\text{hid}}$, where $\mathcal{V}_{\text{vis}}$ is the set of visible nodes and $\mathcal{V}_{\text{hid}}$ is the set of hidden nodes. Then, the total input to hidden (visible) node $v_{i}$ at turn $t$ denoted as $I_{h,i}^{(t)}$ (and $I_{v,i}^{(t)}$) is given by:
$$
\begin{align*}
I_{h,i}^{(t)} &= \sum_{j\in\mathcal{V}} w_{ij}s_{v,j}^{(t-1)} + b_{h,i}\quad\forall i\in\mathcal{V}_{\text{hid}} \\
I_{v,i}^{(t)} &= \sum_{j\in\mathcal{V}} w_{ij}s_{h,j}^{(t-1)} + b_{v,i}\quad\forall i\in\mathcal{V}_{\text{vis}} \\
\end{align*}
$$
where $w_{ij}$ is the weight of the edge connecting $v_{i}$ and $v_{j}$, and $s_{\star,j}^{(t-1)}$ is the state of node $v_{j}$ in layer $\star = \{v,h\}$ at turn $t-1$. Like a full Boltzmann Machine, the state of each node in a restricted Boltzmann Machine is updated stochastically. The probability that node $v_{i}$ is `on` at turn $t$ is given by the logistic function:
$$
\begin{align*}
P(s_{\star,i}^{(t)} = 1 \mid {s}_{\lnot{\star},j}) & = \frac{\exp\left(-\beta\,E(s_{\star,i} = 1 \mid {s}_{\lnot{\star},j})\right)}{\exp\left(-\beta\,E(s_{\star,i} = 1 \mid {s}_{\lnot{\star},j})\right) + \exp\left(-\beta\,E(s_{\star,i} = -1 \mid {s}_{\lnot{\star},j})\right)} \\
\end{align*}
$$
However, we can simplify these probability expressions by substituting in the energy function:
$$
\begin{align*}
E(s_{\star,i}^{(t)} = 1 \mid {s}_{\lnot{\star},i}) & = -s_{\star,i}^{(t)}I_{\star,i}^{(t)} \\
\end{align*}
$$
which gives:
$$
\begin{align*}
P(s_{\star,i}^{(t)} = 1 \mid s_{\lnot{\star},j}) & = \frac{\exp\left(\beta\,I_{\star,i}^{(t)}\right)}{\exp\left(\beta\,I_{\star,i}^{(t)}\right) + \exp\left(-\beta\,I_{\star,i}^{(t)}\right)} \\
& = \frac{1}{1 + \exp(-2\;\beta\,I_{\star,i}^{(t)})}\quad\blacksquare
\end{align*}
$$
where $P(s_{\star,i}^{(t)} = 1 \mid s_{\lnot{\star},i})$ is the probability that node $v_{i}$ is `on` at time $t$ given the state of all the nodes __not__ in layer $\star$ (which we write in compact form as $\lnot\star$). The probability that node $v_{i}$ is `off` at time $t$ is given by $P(s_{i}^{(t)} = -1| s_{\lnot{i}}) = 1 - P(s_{i}^{(t)} = 1 \mid s_{\lnot{i}})$,  i.e., one minus the probability that the node is `on`.
* _What is β_? The parameter $\beta$ is the (inverse) temperature parameter that controls the amount of randomness in the system. As $\beta\rightarrow\infty$, the Boltzmann Machine becomes more deterministic; however, as $\beta\rightarrow{0}$, the Boltzmann Machine becomes more random.
* _What is $s_{\lnot{i}}$?_ This notation refers to the state of all nodes in the network _except_ for node $v_{i}$. In other words, it is the system's state excluding the state of node $v_{i}$. Notice we have not included a superscript $t$ on $s_{\lnot{i}}$. However, from the perspective of node $v_{i}$, the state of the system is always the state of the system at the previous turn, i.e., $s_{\lnot{i}} = \left\{s_{j}^{(t-1)}\right\}_{j\in\mathcal{V}\setminus\{i\}}$.

### Sampling a Restricted Boltzmann Machine (Gibbs Sampling)
To generate samples from a Boltzmann Machine, let us consider the following algorithm: 

__Initialize__ the weights $\mathbf{W}$ and biases $\mathbf{b}$ of the Boltzmann Machine. Provide an initial state $\mathbf{s}^{(0)}$ of the network, a system (inverse) temperature $\beta$, and the number of turns $T$ to run the sampling algorithm.

For each turn $t=1,2,\dots,T$:
1. For each node $v_{i}\in\mathcal{V}$:
    - Compute the total input $h_{i}^{(t)}$ to node $v_{i}$ using the expression: $h_{i}^{(t)} = \sum_{j\in\mathcal{V}} w_{ij}s_{j}^{(t-1)} + b_{i}$.
    - Compute the probability of the _next_ state $s_{i}^{(t)} = 1$ using the logistic function $P(s_{i}^{(t)} = 1 \mid s_{\lnot{i}}) = \left(1+\exp(-2\beta{h}_{i}^{(t)})\right)^{-1}$ for node $v_{i}$. The probability of $s_{i}^{(t)} = -1$ is given by $P(s_{i}^{(t)} = -1| s_{\lnot{i}}) = 1 - P(s_{i}^{(t)} = 1 \mid s_{\lnot{i}})$.
    - Sample the _next_ state of node $v_{i}$ from a [Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution) with parameter $p = P(s_{i}^{(t)} = 1 \mid s_{\lnot{i}})$.
2. Store the state vector $\mathbf{s}^{(t)}$ of the network at turn $t$, and proceed to the next turn.

___

## Training Restricted Boltzmann Machines
Training an RBM involves adjusting the weights and biases to maximize the likelihood of the observed data.